# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df_raw = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_target = pd.read_csv('../data/dayofweek.csv')
df_raw['dayofweek'] = df_target['dayofweek']

In [3]:
X = df_raw.drop(columns=['dayofweek'])
y = df_raw['dayofweek']

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [6]:
svm_params = {
    'C': 10,
    'class_weight': None,
    'gamma': 'auto',
    'kernel': 'rbf',
    'probability': True,
    'random_state': 21
}
tree_params = {
    'class_weight': 'balanced',
    'criterion': 'gini',
    'max_depth': 21,
    'random_state': 21
}
rf_params = {
    'class_weight': 'balanced',
    'criterion': 'entropy',
    'max_depth': 24,
    'n_estimators': 100,
    'random_state': 21
}

In [7]:
svm_clf = SVC(**svm_params)
tree_clf = DecisionTreeClassifier(**tree_params)
rf_clf = RandomForestClassifier(**rf_params)

In [8]:
svm_clf.fit(X_train, y_train)
tree_clf.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=24, random_state=21)

In [9]:
def print_metrics(model, X_val, y_val, name):
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='weighted')
    rec = recall_score(y_val, y_pred, average='weighted')
    print(f"{name}:")
    print(f"  accuracy is {acc:.5f}")
    print(f"  precision is {prec:.5f}")
    print(f"  recall is {rec:.5f}")

In [10]:
print("Individual classifiers on validation set:")
print_metrics(svm_clf, X_valid, y_valid, "SVM")
print_metrics(tree_clf, X_valid, y_valid, "Decision Tree")
print_metrics(rf_clf, X_valid, y_valid, "Random Forest")

Individual classifiers on validation set:
SVM:
  accuracy is 0.87778
  precision is 0.88162
  recall is 0.87778
Decision Tree:
  accuracy is 0.86667
  precision is 0.87170
  recall is 0.86667
Random Forest:
  accuracy is 0.89630
  precision is 0.89698
  recall is 0.89630


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [11]:
estimators = [
    ('svm', svm_clf),
    ('tree', tree_clf),
    ('rf', rf_clf)
]

In [12]:
voting_results = []
for weights in [(1,1,1), (2,1,1), (1,2,1), (1,1,2), (3,1,1), (1,3,1), (1,1,3), (4,1,4)]:
    voting_clf = VotingClassifier(estimators, voting='soft', weights=weights)
    voting_clf.fit(X_train, y_train)
    y_pred = voting_clf.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    rec = recall_score(y_valid, y_pred, average='weighted')
    voting_results.append({
        'weights': weights,
        'accuracy': acc,
        'precision': prec,
        'recall': rec
    })

In [13]:
voting_df = pd.DataFrame(voting_results)
voting_df = voting_df.sort_values(['accuracy', 'precision'], ascending=[False, False])
best_voting = voting_df.iloc[0]
print("Best voting classifier (validation):")
print(f"  weights = {best_voting['weights']}")
print(f"  accuracy = {best_voting['accuracy']:.5f}")
print(f"  precision = {best_voting['precision']:.5f}")
print(f"  recall = {best_voting['recall']:.5f}")

Best voting classifier (validation):
  weights = (4, 1, 4)
  accuracy = 0.91111
  precision = 0.91288
  recall = 0.91111


In [14]:
best_voting_clf = VotingClassifier(estimators, voting='soft', weights=best_voting['weights'])
best_voting_clf.fit(X_train_full, y_train_full)
y_pred_test = best_voting_clf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test, average='weighted')
test_rec = recall_score(y_test, y_pred_test, average='weighted')
print("\nBest voting classifier on test set:")
print(f"  accuracy = {test_acc:.5f}")
print(f"  precision = {test_prec:.5f}")
print(f"  recall = {test_rec:.5f}\n")


Best voting classifier on test set:
  accuracy = 0.92604
  precision = 0.92813
  recall = 0.92604



## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [42]:
bagging_results = []
n_estimators_list = [10, 20, 30, 50, 70, 100]
for n in n_estimators_list:
    bagging_clf = BaggingClassifier(
        base_estimator=SVC(**svm_params),
        n_estimators=n,
        random_state=21,
        n_jobs=-1
    )
    bagging_clf.fit(X_train, y_train)
    y_pred = bagging_clf.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    rec = recall_score(y_valid, y_pred, average='weighted')
    bagging_results.append({
        'n_estimators': n,
        'accuracy': acc,
        'precision': prec,
        'recall': rec
    })

In [43]:
bagging_df = pd.DataFrame(bagging_results)
bagging_df = bagging_df.sort_values(['accuracy', 'precision'], ascending=[False, False])
print(bagging_df)
best_bagging = bagging_df.iloc[0]
print("Best bagging classifier (validation):")
print(f"  n_estimators = {best_bagging['n_estimators']}")
print(f"  accuracy = {best_bagging['accuracy']:.5f}")
print(f"  precision = {best_bagging['precision']:.5f}")
print(f"  recall = {best_bagging['recall']:.5f}")

   n_estimators  accuracy  precision    recall
2            30  0.888889   0.897182  0.888889
0            10  0.885185   0.894269  0.885185
4            70  0.885185   0.893961  0.885185
5           100  0.885185   0.893961  0.885185
1            20  0.885185   0.892576  0.885185
3            50  0.881481   0.890354  0.881481
Best bagging classifier (validation):
  n_estimators = 30.0
  accuracy = 0.88889
  precision = 0.89718
  recall = 0.88889


In [44]:
best_bagging_clf = BaggingClassifier(
    base_estimator=SVC(**svm_params),
    n_estimators=int(best_bagging['n_estimators']),
    random_state=21,
    n_jobs=-1
)
best_bagging_clf.fit(X_train_full, y_train_full)
y_pred_test = best_bagging_clf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test, average='weighted')
test_rec = recall_score(y_test, y_pred_test, average='weighted')
print("\nBest bagging classifier on test set:")
print(f"  accuracy = {test_acc:.5f}")
print(f"  precision = {test_prec:.5f}")
print(f"  recall = {test_rec:.5f}\n")


Best bagging classifier on test set:
  accuracy = 0.89941
  precision = 0.90351
  recall = 0.89941



## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [18]:
stacking_results = []
for n_splits in [2, 3, 4, 5, 6, 7]:
    for passthrough in [False, True]:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        stacking_clf = StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(solver='liblinear'),
            cv=cv,
            passthrough=passthrough,
            n_jobs=-1
        )
        stacking_clf.fit(X_train, y_train)
        y_pred = stacking_clf.predict(X_valid)
        acc = accuracy_score(y_valid, y_pred)
        prec = precision_score(y_valid, y_pred, average='weighted')
        rec = recall_score(y_valid, y_pred, average='weighted')
        stacking_results.append({
            'n_splits': n_splits,
            'passthrough': passthrough,
            'accuracy': acc,
            'precision': prec,
            'recall': rec
        })

In [19]:
stacking_df = pd.DataFrame(stacking_results)
stacking_df = stacking_df.sort_values(['accuracy', 'precision'], ascending=[False, False])
best_stacking = stacking_df.iloc[0]
print("Best stacking classifier (validation):")
print(f"  n_splits = {best_stacking['n_splits']}, passthrough = {best_stacking['passthrough']}")
print(f"  accuracy = {best_stacking['accuracy']:.5f}")
print(f"  precision = {best_stacking['precision']:.5f}")
print(f"  recall = {best_stacking['recall']:.5f}")

Best stacking classifier (validation):
  n_splits = 3, passthrough = True
  accuracy = 0.90741
  precision = 0.91070
  recall = 0.90741


In [20]:
cv_best = StratifiedKFold(n_splits=int(best_stacking['n_splits']), shuffle=True, random_state=21)
best_stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(solver='liblinear'),
    cv=cv_best,
    passthrough=best_stacking['passthrough'],
    n_jobs=-1
)
best_stacking_clf.fit(X_train_full, y_train_full)
y_pred_test = best_stacking_clf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test, average='weighted')
test_rec = recall_score(y_test, y_pred_test, average='weighted')
print("\nBest stacking classifier on test set:")
print(f"  accuracy = {test_acc:.5f}")
print(f"  precision = {test_prec:.5f}")
print(f"  recall = {test_rec:.5f}\n")


Best stacking classifier on test set:
  accuracy = 0.93787
  precision = 0.93943
  recall = 0.93787



## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [21]:
models_performance = [
    ('Voting', best_voting['accuracy'], best_voting['precision'], best_voting_clf),
    ('Bagging', best_bagging['accuracy'], best_bagging['precision'], best_bagging_clf),
    ('Stacking', best_stacking['accuracy'], best_stacking['precision'], best_stacking_clf)
]

In [22]:
best_ensemble = max(models_performance, key=lambda x: (x[1], x[2]))
best_name, best_val_acc, best_val_prec, best_model = best_ensemble
print(f"Best ensemble model: {best_name}")
print(f"  Validation accuracy: {best_val_acc:.5f}, precision: {best_val_prec:.5f}")

Best ensemble model: Voting
  Validation accuracy: 0.91111, precision: 0.91288


In [23]:
if best_name == 'Voting':
    final_model = best_voting_clf
elif best_name == 'Bagging':
    final_model = best_bagging_clf
else:
    final_model = best_stacking_clf

In [24]:
y_pred_final = final_model.predict(X_test)
final_acc = accuracy_score(y_test, y_pred_final)
final_prec = precision_score(y_test, y_pred_final, average='weighted')
final_rec = recall_score(y_test, y_pred_final, average='weighted')
print(f"\nFinal test performance of best ensemble ({best_name}):")
print(f"  accuracy = {final_acc:.5f}")
print(f"  precision = {final_prec:.5f}")
print(f"  recall = {final_rec:.5f}")


Final test performance of best ensemble (Voting):
  accuracy = 0.92604
  precision = 0.92813
  recall = 0.92604


In [25]:
errors = (y_pred_final != y_test)
error_df = X_test[errors].copy()
error_df['true'] = y_test[errors]
error_df['pred'] = y_pred_final[errors]

In [26]:
error_weekday = error_df.groupby('true').size()
total_weekday = y_test.value_counts()
error_pct_weekday = (error_weekday / total_weekday * 100).fillna(0)
print("\nError percentage per weekday (0=Monday,...,6=Sunday):")
print(error_pct_weekday.sort_values(ascending=False))


Error percentage per weekday (0=Monday,...,6=Sunday):
0    22.222222
5     7.407407
1     7.272727
6     7.042254
2     6.666667
4     4.761905
3     3.750000
dtype: float64


In [27]:
labname_cols = [col for col in X.columns if col.startswith('labname_')]
error_labname = {}
total_labname = {}
for col in labname_cols:
    total_labname[col] = (X_test[col] == 1).sum()
    error_labname[col] = (error_df[col] == 1).sum()
error_pct_labname = pd.Series({
    col.replace('labname_', ''): (error_labname[col] / total_labname[col] * 100) if total_labname[col] > 0 else 0
    for col in labname_cols
}).sort_values(ascending=False)
print("\nError percentage per labname:")
print(error_pct_labname)


Error percentage per labname:
lab03       100.000000
laba06       22.222222
laba04       20.000000
lab05s       16.666667
laba04s      12.000000
code_rvw      7.692308
laba06s       6.666667
project1      4.838710
lab02         0.000000
lab03s        0.000000
laba05        0.000000
dtype: float64


In [28]:
user_cols = [col for col in X.columns if col.startswith('uid_user_')]
error_user = {}
total_user = {}
for col in user_cols:
    total_user[col] = (X_test[col] == 1).sum()
    error_user[col] = (error_df[col] == 1).sum()
error_pct_user = pd.Series({
    col.replace('uid_', ''): (error_user[col] / total_user[col] * 100) if total_user[col] > 0 else 0
    for col in user_cols
}).sort_values(ascending=False)
print("\nError percentage per user (top 10):")
print(error_pct_user.head(10))


Error percentage per user (top 10):
user_6     50.000000
user_17    28.571429
user_16    20.000000
user_27    16.666667
user_18    16.666667
user_3     14.285714
user_30    12.500000
user_13    11.764706
user_4     11.111111
user_19    10.526316
dtype: float64


In [29]:
joblib.dump(final_model, '../data/best_ensemble_model.joblib')
print("\nBest ensemble model saved to '../data/best_ensemble_model.joblib'")


Best ensemble model saved to '../data/best_ensemble_model.joblib'
